## 1. Installation des dépendances

In [ ]:
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless kaggle

## 2. Monter Google Drive

On stocke le dataset et les checkpoints sur Drive : Colab réinitialise le disque local à chaque session, Drive persiste.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/segmentation-unet-deeplab'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/experiments', exist_ok=True)
print('Dossier projet sur Drive :', PROJECT_DIR)

## 3. Téléchargement du dataset ISIC 2016 — Task 1 (segmentation)

Dataset le plus léger des challenges ISIC de segmentation, licence CC-0.
Voir `data/README.md` pour le détail et les alternatives (2017, 2018).

In [ ]:
import os

RAW_DIR = f'{PROJECT_DIR}/data/raw'
os.makedirs(f'{RAW_DIR}/images', exist_ok=True)
os.makedirs(f'{RAW_DIR}/masks', exist_ok=True)

%cd {RAW_DIR}
!wget -q https://isic-archive.s3.amazonaws.com/challenges/2016/ISBI2016_ISIC_Part1_Training_Data.zip
!wget -q https://isic-archive.s3.amazonaws.com/challenges/2016/ISBI2016_ISIC_Part1_Training_GroundTruth.zip
!unzip -q ISBI2016_ISIC_Part1_Training_Data.zip -d images_tmp
!unzip -q ISBI2016_ISIC_Part1_Training_GroundTruth.zip -d masks_tmp
!find images_tmp -name '*.jpg' -exec mv {} images/ \;
!find masks_tmp -name '*.png' -exec mv {} masks/ \;
!rm -rf images_tmp masks_tmp *.zip
print('Téléchargement terminé.')

## 4. Premier contact : structure et exploration

In [ ]:
import glob

IMAGES_DIR = f'{PROJECT_DIR}/data/raw/images'
MASKS_DIR = f'{PROJECT_DIR}/data/raw/masks'

image_paths = sorted(glob.glob(f'{IMAGES_DIR}/*'))
mask_paths = sorted(glob.glob(f'{MASKS_DIR}/*'))

print(f"Nombre d'images : {len(image_paths)}")
print(f"Nombre de masques : {len(mask_paths)}")
assert len(image_paths) == len(mask_paths), "Le nombre d'images et de masques ne correspond pas !"

In [ ]:
import cv2
import numpy as np

# Vérifier la variété des tailles d'images (important pour le choix du resize)
sizes = set()
for path in image_paths[:50]:
    img = cv2.imread(path)
    sizes.add(img.shape[:2])

print("Tailles rencontrées (sur 50 images) :", sizes)

In [ ]:
import matplotlib.pyplot as plt

# Visualiser quelques paires image / masque
fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for i in range(3):
    img = cv2.cvtColor(cv2.imread(image_paths[i]), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_paths[i], cv2.IMREAD_GRAYSCALE)
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Image')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title('Masque')
    axes[i, 1].axis('off')
plt.tight_layout()
plt.show()

## 5. Vérifier le déséquilibre de classes

C'est la mesure qui justifiera l'étude d'ablation sur la loss (Dice vs CE vs combo).

In [ ]:
total_pixels = 0
lesion_pixels = 0

for path in mask_paths[:200]:  # échantillon pour aller vite
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    total_pixels += mask.size
    lesion_pixels += (mask > 127).sum()

ratio = lesion_pixels / total_pixels
print(f"Proportion de pixels 'lésion' : {ratio:.2%}")
print(f"Proportion de pixels 'fond'   : {1 - ratio:.2%}")

## Prochaine étape

Une fois ce premier contact validé (dataset accessible, tailles cohérentes, déséquilibre confirmé), on passe à :
- l'écriture du `Dataset` PyTorch avec augmentation (albumentations),
- puis l'implémentation de U-Net.